# 18 Dialogue Collection


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import pandas as pd

import build
import run
from run import dialogue_path
from utils import read_lines
from settings import (BENCHMARK_PATH, DIALOGUE, DIALOGUE_DIR, METHODS,
                      MODELS, PLAN_PATH, PROMPTS_PATH, SEED,
                      WITHHELD_PATH)

pd.set_option('display.width', 200, 'display.max_columns', 40)

MODEL = 'gpt-5.6-luna'

## Design


In [ ]:
print('scenarios        ', DIALOGUE['scenarios'])
print('strata           ', ', '.join(DIALOGUE['strata']))
print('conditions       ', len(DIALOGUE['conditions']),
      '  ' + ', '.join(DIALOGUE['conditions']))
print('methods          ', len(DIALOGUE['methods']))
print('opening replicate', DIALOGUE['opening_replicate'])
print('seed             ', SEED)

for method in DIALOGUE['methods']:
    print(f'\n{method}')
    for turn in METHODS[method]['turns']:
        print(f'   {turn}')

planned = (DIALOGUE['scenarios'] * len(DIALOGUE['conditions'])
           * len(DIALOGUE['methods']))
print(f'\n{planned:,} dialogue texts'
      f'   {planned * len(MODELS):,} dialogues'
      f'   {planned * len(MODELS) * 2:,} replies planned')

## Dialogue Plan


In [ ]:
build.build_turns()

dialogues = pd.read_csv(PLAN_PATH, dtype=str)
print(f'\n{dialogues["dialogue_id"].nunique():,} dialogues, '
      f'{len(dialogues):,} rows')

### Plan Check


In [ ]:
failed = []

# Validation
plan = pd.read_csv(PLAN_PATH, dtype=str)
opening = plan[plan['turn'] == '2']
distinct = opening.groupby('prompt_id')['text'].nunique()
models = opening.groupby('prompt_id')['model'].nunique()
shared = distinct[distinct < models]

print('openings per prompt_id, distinct replies:')
print(distinct.value_counts().sort_index().to_string())
if len(shared):
    failed.append(f'{len(shared)} prompt_ids share an opening across models, '
                  f'the merge on prompt_id and model is wrong')

# Withheld Replies
built = opening.groupby('model').size().rename('dialogues')
print('\n' + built.to_string())

# Withheld Replies

withheld = pd.read_csv(WITHHELD_PATH, dtype=str)
print(f'\n{len(withheld)} openings withheld in the adaptation pass')
print(withheld['model'].value_counts().to_string())

lost = int(built.max()) * len(MODELS) - int(built.sum())
print(f'{lost} of them fall inside the chosen scenarios and conditions')

# Validation
expected = (DIALOGUE['scenarios'] * len(DIALOGUE['conditions'])
            * len(DIALOGUE['methods']) * len(MODELS))
print(f'\n{int(built.sum()):,} dialogues built, {expected:,} planned, '
      f'{int(built.sum()) * 2:,} replies to generate')

for turns in [plan.groupby('dialogue_id').size()]:
    if not (turns == 6).all():
        failed.append(f'{int((turns != 6).sum())} dialogues do not have 6 turns')

if failed:
    raise AssertionError('\n'.join(failed))
print('\nplan is sound, safe to collect')

## Collection


In [ ]:
# Run Commands






### Merge


In [ ]:
for entry in MODELS.values():
    m = entry['id']
    path = dialogue_path(m)
    d = read_lines(path)
    turns = d.groupby('turn').size().to_dict() if not d.empty else {}
    log = path.with_suffix('.log.jsonl').exists()
    dupes = 0 if d.empty else int((d.groupby(['dialogue_id','turn']).size() > 1).sum())
    print(f'{m:<28}{len(d):>6} rows  {turns}  log:{log}  dupes:{dupes}')

In [ ]:
# Blocked Replies











plan = pd.read_csv(PLAN_PATH, dtype=str, keep_default_na=False)

collected = []
for entry in MODELS.values():
    frame = read_lines(dialogue_path(entry['id']))
    if not frame.empty:
        collected.append(frame.assign(turn=frame['turn'].astype(str)))
collected = pd.concat(collected, ignore_index=True)

empty = collected[collected['text'].astype(str).str.strip() == '']

withheld = (empty.merge(
        plan[['dialogue_id', 'turn', 'scenario_id', 'condition', 'method']],
        on=['dialogue_id', 'turn'], how='left')
    .assign(reason='provider filter, no content returned')
    [['dialogue_id', 'scenario_id', 'condition', 'method', 'model', 'turn',
      'reason']]
    .sort_values(['model', 'scenario_id', 'condition']))

path = DIALOGUE_DIR / 'withheld_turns.csv'
withheld.to_csv(path, index=False)

print(f'{len(withheld)} turns withheld inside a dialogue, '
      f'{withheld["dialogue_id"].nunique()} dialogues affected')
print(withheld.to_string(index=False))
print('\nby condition:')
print(withheld['condition'].value_counts().to_string())
print(f'\nwritten to {path.name}')

In [ ]:
# Blocked Replies








from backends import generate

ATTEMPTS = 3
probe_path = DIALOGUE_DIR / 'probe_withheld.jsonl'
probe_path.unlink(missing_ok=True)

withheld = pd.read_csv(DIALOGUE_DIR / 'withheld_turns.csv', dtype=str)
plan = pd.read_csv(PLAN_PATH, dtype=str, keep_default_na=False)

rows = []
for _, want in withheld.iterrows():
    turns = plan[plan['dialogue_id'] == want['dialogue_id']].copy()
    turns['turn'] = turns['turn'].astype(int)
    turns = turns.sort_values('turn')

    # Turns
    
    have = read_lines(dialogue_path(want['model']))
    done = {} if have.empty else {
        str(r['turn']): r['text'] for _, r in
        have[have['dialogue_id'] == want['dialogue_id']].iterrows()}
    turns['text'] = [done.get(str(r['turn']), r['text'])
                     for _, r in turns.iterrows()]

    messages = run.history_before(turns, int(want['turn']))

    for attempt in range(1, ATTEMPTS + 1):
        try:
            text, error = generate('api', want['model'], messages), ''
        except Exception as failure:
            text, error = '', f'{type(failure).__name__}: {failure}'
        rows.append({**want.to_dict(), 'attempt': attempt,
                     'returned': bool(str(text).strip()),
                     'words': len(str(text).split()), 'error': error,
                     'text': text})
        print(f"{want['scenario_id']:<8}{want['condition']:<8}"
              f"turn {want['turn']}  attempt {attempt}  "
              f"{'content' if str(text).strip() else 'withheld'}"
              f"{'  ' + error[:60] if error else ''}")

probe = pd.DataFrame(rows)
probe.to_json(probe_path, orient='records', lines=True)

print(f'\n{len(probe)} calls, {int(probe["returned"].sum())} returned content')
print(probe.groupby(['scenario_id', 'condition', 'turn'])['returned']
      .agg(['sum', 'count']).rename(columns={'sum': 'returned', 'count': 'tried'})
      .to_string())
print(f'\nwritten to {probe_path.name}')

## Yield


In [ ]:
collected = read_lines(dialogue_path(MODEL))

blank = collected['text'].astype(str).str.strip() == ''
print(f'turns returned  {len(collected):,}')
print(f'empty           {int(blank.sum()):,}')
print(f'usable          {int((~blank).sum()):,}')

# Methods






plan = pd.read_csv(PLAN_PATH, dtype=str)
labelled = (collected.assign(turn=collected['turn'].astype(str))
            .merge(plan[['dialogue_id', 'turn', 'method', 'condition']],
                   on=['dialogue_id', 'turn'], how='left'))

print(f'\nunmatched rows: {int(labelled["method"].isna().sum())}')
print('\nby turn:')
print(labelled.groupby('turn').size().rename('replies').to_string())
print('\nby method:')
print(labelled.groupby('method').size().rename('replies').to_string())

## Next Steps
